In [8]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn

In [9]:
import pandas as pd
import numpy as np
import matplotlib as plt
%matplotlib inline
import seaborn as sns


In [14]:
df= pd.read_excel(r'C:\Users\sitar\Downloads\Amazon_Sales_Data\Amazon 2_Raw.xlsx')

In [15]:
df.head()

,Order ID,Order Date,Ship Date,EmailID,Geography,Category,Product Name,Sales,Quantity,Profit
0,CA-2013-138688,2013-06-13,2013-06-17,DarrinVanHuff@gmail.com,"United States,Los Angeles,California",Labels,Self-Adhesive Address Labels for Typewriters b...,14.620,2,6.8714
1,CA-2011-115812,2011-06-09,2011-06-14,BrosinaHoffman@gmail.com,"United States,Los Angeles,California",Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.860,7,14.1694
2,CA-2011-115812,2011-06-09,2011-06-14,BrosinaHoffman@gmail.com,"United States,Los Angeles,California",Art,Newell 322,7.280,4,1.9656
3,CA-2011-115812,2011-06-09,2011-06-14,BrosinaHoffman@gmail.com,"United States,Los Angeles,California",Phones,Mitel 5320 IP Phone VoIP phone,907.152,4,90.7152
4,CA-2011-115812,2011-06-09,2011-06-14,BrosinaHoffman@gmail.com,"United States,Los Angeles,California",Binders,DXL Angle-View Binders with Locking Rings by S...,18.504,3,5.7825


In [16]:
df=df.drop_duplicates()

In [17]:
print(df.isnull().sum())

Order ID        0
Order Date      0
Ship Date       0
EmailID         0
Geography       0
Category        0
Product Name    0
Sales           0
Quantity        0
Profit          0
dtype: int64


In [32]:
df.columns = df.columns.str.replace(" ","")

df.columns =[
    'order_id','order_date','ship_date','email_id','geography','category','product_name','sales','quantity','profit'
]

In [33]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit'],
      dtype='str')


In [36]:
df["order_date"] = pd.to_datetime(df["order_date"], errors ="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors ="coerce")

df["sales"] = pd.to_numeric(df["sales"], errors ="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors ="coerce")
df["profit"] = pd.to_numeric(df["profit"], errors ="coerce")

In [37]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit'],
      dtype='str')


In [38]:
df =df[df["ship_date"] >= df["order_date"]]

In [39]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.month_name()
df["order_day"] = df["order_date"].dt.day

In [40]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day'],
      dtype='str')


In [41]:
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days

In [42]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day', 'shipping_days'],
      dtype='str')


In [43]:
df["profit_margin"] = (df["profit"] / df["sales"]) * 100
df["profit_margin"] = df["profit_margin"].round(2)

In [45]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day', 'shipping_days',
       'profit_margin'],
      dtype='str')


In [46]:
df["avg_price_per_unit"] = df["sales"] / df["quantity"]

In [47]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day', 'shipping_days',
       'profit_margin', 'avg_price_per_unit'],
      dtype='str')


In [48]:
# Identify repeat customers
df["customer_type"] = df.duplicated(subset=["email_id"], keep=False)
df["customer_type"] = df["customer_type"].map({True: "repeat", False: "new"})

In [49]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day', 'shipping_days',
       'profit_margin', 'avg_price_per_unit', 'customer_type'],
      dtype='str')


In [50]:
# Loss making orders
df["loss_flag"] = df["profit"] < 0

# High value orders
df["high_value_order"] = df["sales"] > df["sales"].quantile(0.75)

In [51]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day', 'shipping_days',
       'profit_margin', 'avg_price_per_unit', 'customer_type', 'loss_flag',
       'high_value_order'],
      dtype='str')


In [52]:
# Remove extreme sales outliers (example)
q1 = df["sales"].quantile(0.25)
q3 = df["sales"].quantile(0.75)
iqr = q3 - q1

df = df[(df["sales"] >= q1 - 1.5*iqr) & (df["sales"] <= q3 + 1.5*iqr)]

In [53]:
print(df.columns)

Index(['order_id', 'order_date', 'ship_date', 'email_id', 'geography',
       'category', 'product_name', 'sales', 'quantity', 'profit', 'order_year',
       'order_month', 'month_name', 'order_day', 'shipping_days',
       'profit_margin', 'avg_price_per_unit', 'customer_type', 'loss_flag',
       'high_value_order'],
      dtype='str')


In [55]:
print(df.info)


<bound method DataFrame.info of             order_id order_date  ship_date                  email_id  \
0     CA-2013-138688 2013-06-13 2013-06-17   DarrinVanHuff@gmail.com   
1     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
2     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
4     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
5     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
...              ...        ...        ...                       ...   
3198  CA-2013-125794 2013-09-30 2013-10-04     MarisLaWare@gmail.com   
3199  CA-2014-121258 2014-02-27 2014-03-04      DaveBrooks@gmail.com   
3200  CA-2014-121258 2014-02-27 2014-03-04      DaveBrooks@gmail.com   
3201  CA-2014-121258 2014-02-27 2014-03-04      DaveBrooks@gmail.com   
3202  CA-2014-119914 2014-05-05 2014-05-10     ChrisCortes@gmail.com   

                                 geography     category  \
0     United States,Los Angeles,California  

In [56]:
print(df.describe)

<bound method NDFrame.describe of             order_id order_date  ship_date                  email_id  \
0     CA-2013-138688 2013-06-13 2013-06-17   DarrinVanHuff@gmail.com   
1     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
2     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
4     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
5     CA-2011-115812 2011-06-09 2011-06-14  BrosinaHoffman@gmail.com   
...              ...        ...        ...                       ...   
3198  CA-2013-125794 2013-09-30 2013-10-04     MarisLaWare@gmail.com   
3199  CA-2014-121258 2014-02-27 2014-03-04      DaveBrooks@gmail.com   
3200  CA-2014-121258 2014-02-27 2014-03-04      DaveBrooks@gmail.com   
3201  CA-2014-121258 2014-02-27 2014-03-04      DaveBrooks@gmail.com   
3202  CA-2014-119914 2014-05-05 2014-05-10     ChrisCortes@gmail.com   

                                 geography     category  \
0     United States,Los Angeles,California

In [61]:
df.to_excel("Amazon_cleaned_data.xlsx", index= False)